# 📦 Invoice Delay Prediction — Model Training Pipeline

**Team Role:** Model Training & Evaluation  
**Target Variable:** `DelayFlag` (1 = Late Payment, 0 = On-Time)  
**Dataset:** `invoices_clean.csv` (pre-processed by teammate)  
**Feature List:** `feature_names.pkl` (to avoid leakage)

---
### Pipeline Overview
1. Load data + feature names
2. Temporal train/test split (no random shuffle)
3. SMOTE on training data only
4. Train: Logistic Regression, Random Forest, XGBoost
5. Evaluate & compare all models
6. SHAP analysis on best model
7. Save outputs: `model.pkl`, `shap_explainer.pkl`, `metrics.json`

## ⚙️ Cell 0 — Install Required Libraries
Run this once if you haven't installed these packages yet.

In [ ]:
# Run once to install dependencies
# !pip install imbalanced-learn xgboost shap scikit-learn pandas matplotlib seaborn

## 📚 Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

# Class imbalance
from imblearn.over_sampling import SMOTE

# SHAP
import shap

warnings.filterwarnings('ignore')
print('✅ All libraries loaded successfully!')

## 📂 Cell 2 — Load Dataset & Feature Names

We load the cleaned dataset and the pre-approved feature list from our teammate.  
Using `feature_names.pkl` ensures we don't accidentally include leakage columns (e.g., clearing dates, actual overdue days).

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('invoices_clean.csv', parse_dates=['Doc_Date', 'Posting_Date'])
print(f'Dataset shape: {df.shape}')

# ── Load approved feature list ────────────────────────────────────────────────
with open('feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print(f'\n✅ {len(feature_names)} features loaded from feature_names.pkl')
print('Features:', feature_names)

# ── Define X and y ────────────────────────────────────────────────────────────
X = df[feature_names]
y = df['DelayFlag']

print(f'\nX shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'\nClass distribution:\n{y.value_counts()}')
print(f'Imbalance ratio — Late(1): {y.mean():.1%}, On-Time(0): {(1-y.mean()):.1%}')

## 🕐 Cell 3 — Temporal Train-Test Split

**Why temporal split?** — Our data is time-series based (invoice dates over years).  
Using a random split would let the model 'see the future', causing data leakage.  
Instead: older 80% of invoices → **train**, most recent 20% → **test**.

In [ ]:
# Sort by Doc_Date to ensure chronological order
df_sorted = df.sort_values('Doc_Date').reset_index(drop=True)

# Find the 80th percentile cutoff date
split_idx = int(len(df_sorted) * 0.80)
cutoff_date = df_sorted['Doc_Date'].iloc[split_idx]

print(f'Total records  : {len(df_sorted):,}')
print(f'Split index    : {split_idx:,}')
print(f'Cutoff date    : {cutoff_date.date()}')
print(f'Date range     : {df_sorted["Doc_Date"].min().date()} → {df_sorted["Doc_Date"].max().date()}')

# Create train/test masks
train_mask = df_sorted.index < split_idx
test_mask  = df_sorted.index >= split_idx

X_train = df_sorted.loc[train_mask, feature_names]
X_test  = df_sorted.loc[test_mask,  feature_names]
y_train = df_sorted.loc[train_mask, 'DelayFlag']
y_test  = df_sorted.loc[test_mask,  'DelayFlag']

print(f'\nTrain set : {X_train.shape[0]:,} rows ({y_train.mean():.1%} late)')
print(f'Test set  : {X_test.shape[0]:,} rows  ({y_test.mean():.1%} late)')

## ⚖️ Cell 4 — Handle Class Imbalance with SMOTE

SMOTE = **Synthetic Minority Over-sampling TEchnique**  
It creates synthetic samples for the minority class to balance training data.  

⚠️ **Important:** We apply SMOTE **only on training data** — never on test data.  
Applying SMOTE to the test set would give artificially inflated metrics.

In [ ]:
print('Class distribution BEFORE SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nClass distribution AFTER SMOTE:')
print(pd.Series(y_train_res).value_counts())
print(f'\nTrain set size after SMOTE: {X_train_res.shape[0]:,} rows')

## 📏 Cell 5 — Feature Scaling

Logistic Regression is sensitive to feature scale.  
We fit the scaler on training data only and transform both train and test.

In [ ]:
scaler = StandardScaler()

# Fit ONLY on training data — then transform both
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('✅ Scaling complete')
print(f'Train scaled shape : {X_train_scaled.shape}')
print(f'Test scaled shape  : {X_test_scaled.shape}')

## 🤖 Cell 6 — Train All Models

We train three models of increasing complexity:
- **Logistic Regression** — linear baseline, fast, interpretable
- **Random Forest** — ensemble of decision trees, handles non-linearity
- **XGBoost** — gradient boosted trees, often best for tabular data

In [ ]:
# ── Logistic Regression (uses scaled data) ────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train_res)
print('✅ Logistic Regression trained')

# ── Random Forest (tree-based; doesn't need scaling) ──────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_res, y_train_res)
print('✅ Random Forest trained')

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train_res, y_train_res)
print('✅ XGBoost trained')

print('\n🎉 All models trained successfully!')

## 📊 Cell 7 — Evaluate All Models

We evaluate on the **untouched test set** using 5 metrics:
| Metric | What it tells us |
|--------|------------------|
| Accuracy | Overall correct predictions |
| Precision | Of predicted-late, how many were actually late? |
| Recall | Of actual-late, how many did we catch? |
| F1-Score | Balance between Precision & Recall |
| ROC-AUC | Discrimination power across all thresholds |

In [ ]:
def evaluate_model(name, model, X_test_data, y_true):
    """Evaluate a model and return a metrics dictionary."""
    y_pred  = model.predict(X_test_data)
    y_proba = model.predict_proba(X_test_data)[:, 1]

    metrics = {
        'model'    : name,
        'accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred), 4),
        'recall'   : round(recall_score(y_true, y_pred), 4),
        'f1_score' : round(f1_score(y_true, y_pred), 4),
        'roc_auc'  : round(roc_auc_score(y_true, y_proba), 4),
    }
    return metrics, y_pred

# Evaluate each model (LR uses scaled data, others use unscaled)
lr_metrics,  lr_pred  = evaluate_model('Logistic Regression', lr,  X_test_scaled, y_test)
rf_metrics,  rf_pred  = evaluate_model('Random Forest',       rf,  X_test,        y_test)
xgb_metrics, xgb_pred = evaluate_model('XGBoost',             xgb, X_test,        y_test)

all_metrics = [lr_metrics, rf_metrics, xgb_metrics]

# ── Pretty comparison table ───────────────────────────────────────────────────
results_df = pd.DataFrame(all_metrics).set_index('model')
print('='*65)
print('           MODEL PERFORMANCE COMPARISON')
print('='*65)
print(results_df.to_string())
print('='*65)

# Identify best model by ROC-AUC
best_model_name = results_df['roc_auc'].idxmax()
print(f'\n🏆 Best Model (by ROC-AUC): {best_model_name}')

## 🔲 Cell 8 — Confusion Matrices

A confusion matrix shows the breakdown of:
- **True Positives (TP):** Correctly predicted late
- **True Negatives (TN):** Correctly predicted on-time
- **False Positives (FP):** Predicted late, was on-time
- **False Negatives (FN):** Predicted on-time, was actually late ← most costly!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

models_eval = [
    ('Logistic Regression', lr_pred),
    ('Random Forest',       rf_pred),
    ('XGBoost',             xgb_pred),
]

for ax, (name, pred) in zip(axes, models_eval):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['On-Time', 'Late'],
        yticklabels=['On-Time', 'Late']
    )
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrices.png')

## 📈 Cell 9 — ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))

models_proba = [
    ('Logistic Regression', lr,  X_test_scaled, lr_metrics['roc_auc']),
    ('Random Forest',       rf,  X_test,        rf_metrics['roc_auc']),
    ('XGBoost',             xgb, X_test,        xgb_metrics['roc_auc']),
]

for name, model, X_data, auc in models_proba:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_data)[:, 1])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc})', linewidth=2)

plt.plot([0,1], [0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: roc_curves.png')

## 🏆 Cell 10 — Select Best Model & Print Final Report

In [ ]:
# Map model name to object and test data
model_registry = {
    'Logistic Regression': (lr,  X_test_scaled),
    'Random Forest'      : (rf,  X_test),
    'XGBoost'            : (xgb, X_test),
}

best_model_obj, best_X_test = model_registry[best_model_name]

print(f'🏆 Best Model: {best_model_name}')
print(f'   ROC-AUC  : {results_df.loc[best_model_name, "roc_auc"]}')
print(f'   F1-Score : {results_df.loc[best_model_name, "f1_score"]}')
print(f'   Recall   : {results_df.loc[best_model_name, "recall"]}')
print()

best_pred = best_model_obj.predict(best_X_test)
print('Classification Report — Best Model:')
print(classification_report(y_test, best_pred, target_names=['On-Time', 'Late']))

## 🔍 Cell 11 — SHAP Analysis (Feature Importance)

SHAP (SHapley Additive exPlanations) explains **why** the model made each prediction.  
- **Positive SHAP value** → pushes prediction toward *Late (1)*
- **Negative SHAP value** → pushes prediction toward *On-Time (0)*  

We use a sample of 500 rows for efficiency.

In [ ]:
# Use a representative sample for speed
shap_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)

# Choose explainer based on best model type
if best_model_name in ['Random Forest', 'XGBoost']:
    explainer = shap.TreeExplainer(best_model_obj)
    shap_values = explainer.shap_values(shap_sample)
    # For tree models, shap_values can be a list [class0, class1]
    if isinstance(shap_values, list):
        sv = shap_values[1]   # class 1 = Late
    else:
        sv = shap_values
else:
    # Logistic Regression
    shap_sample_scaled = scaler.transform(shap_sample)
    explainer = shap.LinearExplainer(best_model_obj, shap_sample_scaled)
    shap_values = explainer.shap_values(shap_sample_scaled)
    sv = shap_values

print('✅ SHAP values computed')
print(f'SHAP values shape: {np.array(sv).shape}')

### SHAP Summary Plot — Global Feature Importance

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv, shap_sample,
    feature_names=feature_names,
    show=False,
    plot_size=(10, 7)
)
plt.title(f'SHAP Summary — {best_model_name}', fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: shap_summary.png')

### SHAP Bar Plot — Top 15 Most Important Features

In [ ]:
# Mean absolute SHAP values = average impact on model output
mean_shap = np.abs(sv).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature'         : feature_names,
    'Mean_SHAP_Value' : mean_shap
}).sort_values('Mean_SHAP_Value', ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=shap_importance,
    x='Mean_SHAP_Value', y='Feature',
    palette='viridis'
)
plt.title(f'Top 15 Features by Mean |SHAP| — {best_model_name}', fontsize=12)
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: shap_bar.png')
print('\nTop 10 Most Influential Features:')
print(shap_importance.head(10).to_string(index=False))

## 💾 Cell 12 — Save Outputs

We save three files for the next pipeline stage and for reproducibility:
- `model.pkl` → trained best model
- `shap_explainer.pkl` → SHAP explainer object
- `metrics.json` → evaluation results for all models

In [ ]:
import os
os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

# ── 1. Save best model ────────────────────────────────────────────────────────
with open('models/model.pkl', 'wb') as f:
    pickle.dump(best_model_obj, f)
print('✅ Saved: models/model.pkl')

# ── 2. Save scaler (needed to preprocess future inputs for LR) ────────────────
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Saved: models/scaler.pkl')

# ── 3. Save SHAP explainer ────────────────────────────────────────────────────
with open('models/shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print('✅ Saved: models/shap_explainer.pkl')

# ── 4. Save metrics JSON ──────────────────────────────────────────────────────
metrics_output = {
    'best_model': best_model_name,
    'split_info': {
        'method'          : 'temporal',
        'cutoff_date'     : str(cutoff_date.date()),
        'train_size'      : int(X_train.shape[0]),
        'test_size'       : int(X_test.shape[0]),
        'train_size_smote': int(X_train_res.shape[0]),
    },
    'models': all_metrics
}

with open('results/metrics.json', 'w') as f:
    json.dump(metrics_output, f, indent=2)
print('✅ Saved: results/metrics.json')

# ── 5. Move plots to results ──────────────────────────────────────────────────
import shutil
for plot_file in ['confusion_matrices.png', 'roc_curves.png', 'shap_summary.png', 'shap_bar.png']:
    if os.path.exists(plot_file):
        shutil.move(plot_file, f'results/{plot_file}')
        print(f'✅ Moved: results/{plot_file}')

print('\n🎉 All outputs saved successfully!')

## ✅ Cell 13 — Final Summary

In [ ]:
print('='*60)
print('         FINAL PIPELINE SUMMARY')
print('='*60)
print(f'Dataset         : {len(df):,} invoices')
print(f'Features used   : {len(feature_names)} (from feature_names.pkl)')
print(f'Split method    : Temporal (cutoff: {cutoff_date.date()})')
print(f'Train / Test    : {X_train.shape[0]:,} / {X_test.shape[0]:,}')
print(f'After SMOTE     : {X_train_res.shape[0]:,} training samples')
print()
print('Model Results (on test set):')
print(results_df.to_string())
print()
print(f'🏆 Best Model: {best_model_name}')
print(f'   ROC-AUC  : {results_df.loc[best_model_name, "roc_auc"]}')
print(f'   F1-Score : {results_df.loc[best_model_name, "f1_score"]}')
print()
print('Saved Files:')
print('  models/model.pkl            ← Best trained model')
print('  models/scaler.pkl           ← StandardScaler')
print('  models/shap_explainer.pkl   ← SHAP explainer')
print('  results/metrics.json        ← All evaluation metrics')
print('  results/confusion_matrices.png')
print('  results/roc_curves.png')
print('  results/shap_summary.png')
print('  results/shap_bar.png')
print('='*60)